In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import pingouin as pg
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from matplotlib.lines import Line2D
import matplotlib.lines as mlines
import math
import seaborn as sns
from scipy.optimize import curve_fit
import pyreadr

## I. Purpose ##

### Analzye effects of SNPs inside GWAS LD blocks ##

## II. read input data

In [5]:
df_ld_block =pd.read_pickle('/Users/jialexu/Desktop/Project2GWAS-BehvaioralGenetics/experiments/Gwas/data/LD_block/20250525/ld_block_info_20250903.pkl')

## III. Prepare SNP dataset for VEP analysis 

In [7]:
wdir = '/Users/jialexu/Desktop/Project2GWAS-BehvaioralGenetics/experiments/VEP/'

df_vep_input = []
for QTL in df_ld_block.QTL_id:
    
    block_start = df_ld_block.loc[df_ld_block['QTL_id']==QTL,'block_start'].iloc[0]
    block_end = df_ld_block.loc[df_ld_block['QTL_id']==QTL,'block_end'].iloc[0]
    df_snp = df_ld_block.loc[df_ld_block['QTL_id']==QTL,'SNPs_nearby'].iloc[0]
    df_snp = df_snp.loc[(df_snp['pos']>=block_start)&(df_snp['pos']<=block_end)].dropna()
    df_snp = df_snp.loc[:,['chr', 'pos', 'A1','A2','-log10_P', 'r2']]
    df_snp['allele'] = df_snp['A1']+'/'+df_snp['A2']
    df_snp['pos_2'] = df_snp['pos']
    df_snp = df_snp.loc[:,['chr', 'pos', 'pos_2','allele','-log10_P', 'r2']]
    #df_snp['QTL_no'] = QTL
    df_vep_input.append(df_snp)
df_vep_input = pd.concat(df_vep_input)
#df_vep_input.to_csv(wdir+'LD_SNP_VEP_input.txt', header=None, sep='\t', index=None)

In [8]:
df_snp_logp = df_snp_r2=df_vep_input.copy()
df_snp_logp['SNP_id'] = df_snp_logp.index
df_snp_logp = df_snp_logp.sort_values(['SNP_id', '-log10_P'], ascending=False).drop_duplicates(['SNP_id'])
df_snp_r2['SNP_id'] = df_snp_r2.index
df_snp_r2 = df_snp_r2.sort_values(['SNP_id', 'r2'], ascending=False).drop_duplicates(['SNP_id'])

## IV. Perform VEP analysis 

### This step was performed on Ensemble VEP web application and the results were downloaded to current wdir

## V. Analzye VEP results

In [13]:
## read VEP results data from ensembl
df_vep_output = pd.read_csv(wdir+'LD_SNP_VEP_output.txt',sep='\t')
df_vep_output

/var/folders/76/sy0y1dxn0vz_wwg79p6ty5cm0000gn/T/ipykernel_49694/4036104859.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_vep_output = pd.read_csv(wdir+'LD_SNP_VEP_output.txt',sep='\t')


,#Uploaded_variation,Location,Allele,Consequence,IMPACT,SYMBOL,Gene,Feature_type,Feature,BIOTYPE,...,HGNC_ID,MANE,MANE_SELECT,MANE_PLUS_CLINICAL,TSL,APPRIS,SIFT,CLIN_SIG,SOMATIC,PHENO
0,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,brat1,ENSDARG00000062585,Transcript,ENSDART00000090689.4,protein_coding,...,-,-,-,-,-,P1,-,-,-,-
1,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000111041.4,protein_coding,...,-,-,-,-,-,P5,-,-,-,-
2,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000134531.2,protein_coding,...,-,-,-,-,-,A2,-,-,-,-
3,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000142045.2,processed_transcript,...,-,-,-,-,-,-,-,-,-,-
4,1_11690436_C/T,1:11690436-11690436,T,upstream_gene_variant,MODIFIER,brat1,ENSDARG00000062585,Transcript,ENSDART00000090689.4,protein_coding,...,-,-,-,-,-,P1,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708386,8_18192639_G/A,8:18192639-18192639,A,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,-,-,-,-,-,P1,-,-,-,-
708387,8_18192686_G/A,8:18192686-18192686,A,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,-,-,-,-,-,P1,-,-,-,-
708388,8_18192802_A/G,8:18192802-18192802,G,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,-,-,-,-,-,P1,-,-,-,-
708389,8_18192819_A/T,8:18192819-18192819,T,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,-,-,-,-,-,P1,-,-,-,-


In [30]:
## read the classification file
df_vep_class = pd.read_csv(wdir+'SNP_cat_v1.csv')
df_vep_class

,Unnamed: 0,CL1,CL2,CL3,Consequence
0,0,Intergenic,NaN,NaN,upstream_gene_variant
1,1,Intergenic,NaN,NaN,intergenic_variant
2,2,Intergenic,NaN,NaN,downstream_gene_variant
3,3,NC-gene,cds,NP,non_coding_transcript_exon_variant
4,4,NC-gene,cds,splice,"splice_region_variant,non_coding_transcript_ex..."
5,5,NC-gene,cds,splice,"splice_region_variant,intron_variant,non_codin..."
6,6,NC-gene,intron,splice,"splice_donor_region_variant,intron_variant,non..."
7,7,NC-gene,intron,splice,"splice_donor_5th_base_variant,intron_variant,n..."
8,8,NC-gene,intron,splice,"splice_polypyrimidine_tract_variant,intron_var..."
9,9,NC-gene,intron,splice,"splice_region_variant,splice_polypyrimidine_tr..."


In [32]:
##combine the class file with VEP data
df_vep_new = pd.merge(df_vep_output, df_vep_class, on='Consequence')
df_vep_new['SNP_id'] = df_vep_new['Location'].str.split('-').str[0]
df_vep_new

,#Uploaded_variation,Location,Allele,Consequence,IMPACT,SYMBOL,Gene,Feature_type,Feature,BIOTYPE,...,APPRIS,SIFT,CLIN_SIG,SOMATIC,PHENO,Unnamed: 0,CL1,CL2,CL3,SNP_id
0,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,brat1,ENSDARG00000062585,Transcript,ENSDART00000090689.4,protein_coding,...,P1,-,-,-,-,0,Intergenic,NaN,NaN,1:11690229
1,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000111041.4,protein_coding,...,P5,-,-,-,-,0,Intergenic,NaN,NaN,1:11690229
2,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000134531.2,protein_coding,...,A2,-,-,-,-,0,Intergenic,NaN,NaN,1:11690229
3,1_11690229_A/G,1:11690229-11690229,G,upstream_gene_variant,MODIFIER,eif3bb,ENSDARG00000074213,Transcript,ENSDART00000142045.2,processed_transcript,...,-,-,-,-,-,0,Intergenic,NaN,NaN,1:11690229
4,1_11690436_C/T,1:11690436-11690436,T,upstream_gene_variant,MODIFIER,brat1,ENSDARG00000062585,Transcript,ENSDART00000090689.4,protein_coding,...,P1,-,-,-,-,0,Intergenic,NaN,NaN,1:11690436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708386,8_18192639_G/A,8:18192639-18192639,A,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,P1,-,-,-,-,2,Intergenic,NaN,NaN,8:18192639
708387,8_18192686_G/A,8:18192686-18192686,A,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,P1,-,-,-,-,2,Intergenic,NaN,NaN,8:18192686
708388,8_18192802_A/G,8:18192802-18192802,G,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,P1,-,-,-,-,2,Intergenic,NaN,NaN,8:18192802
708389,8_18192819_A/T,8:18192819-18192819,T,downstream_gene_variant,MODIFIER,rps8b,ENSDARG00000057368,Transcript,ENSDART00000080014.5,protein_coding,...,P1,-,-,-,-,2,Intergenic,NaN,NaN,8:18192819


In [34]:
df_vep_logp = pd.merge(df_vep_new, df_snp_logp[['-log10_P', 'SNP_id']], on='SNP_id')
df_vep_r2 = pd.merge(df_vep_new, df_snp_r2[['r2', 'SNP_id']], on='SNP_id')

In [36]:
df_vep_sigsnp = df_vep_logp.loc[df_vep_logp['-log10_P']>6.81]
df_vep_sigsnp.SNP_id.unique().shape[0]

1288

In [40]:
##save VEP results for significant variants for publication
df_vep_sigsnp.to_excel('/Users/jialexu/Desktop/Project2GWAS-BehvaioralGenetics/doc/submission_2025/Temp_tables/VEP_v1.xlsx', index=False)

In [319]:
wdir = '/Users/jialexu/Desktop/Project2GWAS-BehvaioralGenetics/experiments/Database/motif/'
gene_corr = pd.read_csv(wdir+'DEnpgTFs.txt', sep='\t')
gene_corr.columns = ['human_ens', 'start', 'end', 'human_symbol', 'chr','strand']
gene_corr

,human_ens,start,end,human_symbol,chr,strand
0,ENSG00000068323,49028726,49043476,TFE3,X,-1
1,ENSG00000081189,88717117,88904257,MEF2C,5,-1
2,ENSG00000101057,43667019,43716495,MYBL2,20,1
3,ENSG00000102145,48786540,48794311,GATA1,X,1
4,ENSG00000109381,139028112,139177224,ELF2,4,-1
5,ENSG00000124216,49982980,49988886,SNAI1,20,1
6,ENSG00000131668,93951627,93955355,BARX1,9,-1
7,ENSG00000148516,31318495,31529814,ZEB1,10,1
8,ENSG00000162624,75128434,75161533,LHX8,1,1
9,ENSG00000174332,53506232,53739164,GLIS1,1,-1


In [327]:
df_ortho_genes = pd.read_excel('/Users/jialexu/Desktop/Project2GWAS-BehvaioralGenetics/experiments/Database/ZebrafishGenomeAnnotation/ortho_genes_20250227.xlsx', sheet_name = 'human_orthos')
df_ortho_genes.head()

,ZFIN ID,zebrafish_symbol,zebrafish_ens,ZFIN Name,human_symbol,human_ens,Human Name,OMIM ID,Gene ID,HGNC ID
0,ZDB-GENE-000112-47,ppardb,ENSDARG00000009473,peroxisome proliferator-activated receptor del...,PPARD,ENSG00000112033,peroxisome proliferator activated receptor delta,600409.0,5467,9235.0
1,ZDB-GENE-000125-12,igfbp2a,ENSDARG00000052470,insulin-like growth factor binding protein 2a,IGFBP2,ENSG00000115457,insulin like growth factor binding protein 2,146731.0,3485,5471.0
2,ZDB-GENE-000125-4,dlc,ENSDARG00000002336,deltaC,DLL3,ENSG00000090932,delta like canonical Notch ligand 3,602768.0,10683,2909.0
3,ZDB-GENE-000128-11,dbx1b,ENSDARG00000001859,developing brain homeobox 1b,DBX1,ENSG00000109851,developing brain homeobox 1,619830.0,120237,33185.0
4,ZDB-GENE-000128-13,dbx2,ENSDARG00000105053,developing brain homeobox 2,DBX2,ENSG00000185610,developing brain homeobox 2,620706.0,440097,33186.0


In [1169]:
enriched_deg_tf_gene = pyreadr.read_r('/Users/jialexu/Desktop/Project11MultiomeRNAseq/Seurat/GEX_analysis/processed_data/enriched_deg_tf_gene_v1.rds')[None]

In [1171]:
df_temp = df_vep_r2.loc[df_vep_r2.Gene.isin(enriched_deg_tf_gene.loc[enriched_deg_tf_gene['zGWAS_tf']=='y'].zebrafish_ens)]

In [1175]:
df_temp.head()

,#Uploaded_variation,Location,Allele,Consequence,IMPACT,SYMBOL,Gene,Feature_type,Feature,BIOTYPE,...,APPRIS,SIFT,CLIN_SIG,SOMATIC,PHENO,CL1,CL2,CL3,SNP_id,r2
33838,11_25246526_A/G,11:25246526-25246526,G,downstream_gene_variant,MODIFIER,snai1a,ENSDARG00000056995,Transcript,ENSDART00000130477.4,protein_coding,...,P1,-,-,-,-,Intergenic,NaN,NaN,11:25246526,0.003897
33839,11_25246559_T/G,11:25246559-25246559,G,downstream_gene_variant,MODIFIER,snai1a,ENSDARG00000056995,Transcript,ENSDART00000130477.4,protein_coding,...,P1,-,-,-,-,Intergenic,NaN,NaN,11:25246559,0.003897
33840,11_25246691_T/C,11:25246691-25246691,C,downstream_gene_variant,MODIFIER,snai1a,ENSDARG00000056995,Transcript,ENSDART00000130477.4,protein_coding,...,P1,-,-,-,-,Intergenic,NaN,NaN,11:25246691,0.003897
33841,11_25246705_A/G,11:25246705-25246705,G,downstream_gene_variant,MODIFIER,snai1a,ENSDARG00000056995,Transcript,ENSDART00000130477.4,protein_coding,...,P1,-,-,-,-,Intergenic,NaN,NaN,11:25246705,0.003897
33842,11_25246801_A/G,11:25246801-25246801,G,downstream_gene_variant,MODIFIER,snai1a,ENSDARG00000056995,Transcript,ENSDART00000130477.4,protein_coding,...,P1,-,-,-,-,Intergenic,NaN,NaN,11:25246801,0.003897
